# Хакатон: стартовый ноутбук
Илья Новицкий · Блок 6 · хакатон 12 декабря 2026, 12:00–15:10

Ноутбук — каркас для любого трека. Он не решает задачу за вас, а даёт то, что нужно каждой команде
с первого часа:

1. **Единый клиент модели** — облачный API, свой сервер (llama.cpp, vLLM, TGI) или локальная модель.
2. **Журнал вызовов** — задержка, токены, условная стоимость.
3. **Формат набора проверок** и его загрузка.
4. **Базовая линия и ваша система** как две функции `question → answer`.
5. **Оценка**: совпадение с эталоном, попарное сравнение моделью-судьёй с перестановкой, задержка p50/p95, стоимость.
6. **Отчёт** — таблица и `report.json` для README и защиты.

В демонстрации базовая линия — модель без контекста, система — простой RAG на BM25 по статьям
SberQuAD. Замените обе функции своими и подставьте свой набор проверок.

In [ ]:
# requirements (Python 3.10–3.12); в скобках — версии, на которых ноутбук прогнан:
#   openai>=1.40         (проверено 3.46.0)
#   pandas>=2.2          (проверено 3.1.1)
#   numpy>=1.26          (проверено 2.5.4)
#   datasets>=3.0        (проверено 5.1.0)
#   rank-bm25>=0.2.2     (проверено 0.2.2)
#   nltk>=3.9            (проверено 3.10.4)
#   torch>=2.4           (проверено 2.15.0)
#   transformers>=4.46   (проверено 5.22.2)
#   accelerate>=0.34     (проверено 1.15.0)
# Установка локально: pip install -r requirements.txt
# Google Colab: раскомментируйте строку ниже и перезапустите среду после установки.
# %pip install -q "openai>=1.40" "pandas>=2.2" "numpy>=1.26" "datasets>=3.0" "rank-bm25>=0.2.2" "nltk>=3.9" "torch>=2.4" "transformers>=4.46" "accelerate>=0.34"
#
# torch/transformers нужны только для BACKEND="local". Зафиксируйте версии своей команды в requirements.txt решения.

import sys
print("Python", sys.version.split()[0])

## 1. Конфигурация
| `BACKEND` | Что это | Что задать |
|---|---|---|
| `api` | Облачный OpenAI-совместимый API | `NLP_API_BASE_URL`, `OPENAI_API_KEY`, `NLP_API_MODEL` |
| `server` | Свой сервер: llama.cpp, vLLM, TGI, Ollama | `NLP_SERVER_URL` (например, `http://127.0.0.1:8000/v1`), `NLP_SERVER_MODEL` |
| `local` | Модель в этом процессе через transformers | `NLP_LOCAL_MODEL` |

Цены нужны только для оценки стоимости — подставьте тариф своего поставщика или себестоимость сервера.

In [ ]:
import os, re, json, time, random, hashlib
from pathlib import Path
from collections import Counter
from getpass import getpass

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
random.seed(SEED)
TEAM = os.getenv("TEAM_NAME", "demo-team")
BACKEND = os.getenv("NLP_BACKEND", "api")
PRICE_IN_PER_M = float(os.getenv("PRICE_IN_PER_M", "0.25"))     # $ за миллион входных токенов (условно)
PRICE_OUT_PER_M = float(os.getenv("PRICE_OUT_PER_M", "1.25"))   # $ за миллион выходных токенов (условно)
RUN_DIR = Path("hackathon_runs") / time.strftime("%Y%m%d-%H%M%S")
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Команда:", TEAM, "| бэкенд:", BACKEND, "| журнал:", RUN_DIR)

## 2. Клиент модели с журналом
Все вызовы модели в проекте идут через `LLM.chat`. Каждый вызов записывается в `calls.jsonl`:
это пригодится и для отчёта, и для отладки на защите.

In [ ]:
class LLM:
    def __init__(self, backend=BACKEND, log_path=RUN_DIR / "calls.jsonl"):
        self.backend, self.log_path = backend, Path(log_path)
        if backend in ("api", "server"):
            from openai import OpenAI
            if backend == "api":
                base = os.getenv("NLP_API_BASE_URL") or getpass("Базовый адрес API (…/v1): ")
                key = os.getenv("OPENAI_API_KEY") or getpass("API key: ")
                self.model = os.getenv("NLP_API_MODEL", "gpt-6-sol")
            else:
                base, key = os.getenv("NLP_SERVER_URL", "http://127.0.0.1:8000/v1"), "local"
                self.model = os.getenv("NLP_SERVER_MODEL", "local-model")
            self.client = OpenAI(base_url=base.rstrip("/"), api_key=key, timeout=120, max_retries=1)
        else:
            import torch
            from transformers import AutoTokenizer, AutoModelForCausalLM
            self.model = os.getenv("NLP_LOCAL_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
            self.device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
            self.tok = AutoTokenizer.from_pretrained(self.model)
            self.hf = AutoModelForCausalLM.from_pretrained(
                self.model, dtype=torch.float32 if self.device == "cpu" else torch.bfloat16).to(self.device).eval()
            self.torch = torch

    def chat(self, messages, max_tokens=256, temperature=0.0, tag=""):
        t0 = time.perf_counter(); error = None; text = ""; n_in = n_out = None
        try:
            if self.backend in ("api", "server"):
                kwargs = dict(model=self.model, messages=messages)
                kwargs["max_completion_tokens" if self.backend == "api" else "max_tokens"] = max_tokens
                if temperature > 0:
                    kwargs["temperature"] = temperature
                r = self.client.chat.completions.create(**kwargs)
                text = r.choices[0].message.content or ""
                if r.usage:
                    n_in, n_out = r.usage.prompt_tokens, r.usage.completion_tokens
            else:
                prompt = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                enc = self.tok(prompt, return_tensors="pt").to(self.device)
                with self.torch.no_grad():
                    out = self.hf.generate(**enc, max_new_tokens=max_tokens, do_sample=temperature > 0,
                                           temperature=temperature or None, top_p=None, top_k=None,
                                           pad_token_id=self.tok.eos_token_id)
                new = out[0, enc["input_ids"].shape[1]:]
                text = self.tok.decode(new, skip_special_tokens=True)
                n_in, n_out = int(enc["input_ids"].shape[1]), int(len(new))
        except Exception as exc:
            error = f"{type(exc).__name__}: {str(exc)[:200]}"
        record = {"ts": time.time(), "tag": tag, "model": self.model, "latency_s": round(time.perf_counter() - t0, 4),
                  "input_tokens": n_in, "output_tokens": n_out, "error": error,
                  "cost_usd": ((n_in or 0) * PRICE_IN_PER_M + (n_out or 0) * PRICE_OUT_PER_M) / 1e6}
        with open(self.log_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
        if error:
            raise RuntimeError(error)
        return text.strip()

llm = LLM()
print(llm.model, "→", llm.chat([{"role": "user", "content": "Ответь одним словом: готов?"}], max_tokens=5, tag="smoke"))

## 3. Набор проверок
Формат — JSONL, одна задача в строке:

```json
{"id": "q001", "question": "…", "reference": "…", "tags": ["тип вопроса"]}
```

`reference` может быть пустым, если эталона нет (тогда работает только попарная оценка).
**Соберите набор в первые 40 минут** и не меняйте его после начала экспериментов; держите отдельно
несколько примеров для отладки.

Демонстрационный набор — 30 вопросов SberQuAD, база знаний — 200 абзацев русской Википедии, к которым они заданы.

In [ ]:
from datasets import load_dataset
squad = load_dataset("kuznetsoffandrey/sberquad", data_files={"validation": "sberquad/validation-00000-of-00001.parquet"})["validation"].to_pandas()
paragraphs = list(dict.fromkeys(squad["context"])); random.Random(SEED).shuffle(paragraphs)
squad = squad[squad["context"].isin(set(paragraphs[:200]))]
KNOWLEDGE_BASE = list(dict.fromkeys(squad["context"]))      # абзацы-документы
demo = squad.sample(30, random_state=SEED)
with open(RUN_DIR / "eval_set.jsonl", "w", encoding="utf-8") as f:
    for i, (_, r) in enumerate(demo.iterrows()):
        f.write(json.dumps({"id": f"q{i:03d}", "question": r["question"], "reference": r["answers"]["text"][0],
                            "tags": ["sberquad"]}, ensure_ascii=False) + "\n")

def load_eval_set(path):
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    assert len({r["id"] for r in rows}) == len(rows), "id в наборе должны быть уникальны"
    return pd.DataFrame(rows)

EVAL = load_eval_set(RUN_DIR / "eval_set.jsonl")
print("Задач:", len(EVAL), "| документов в базе:", len(KNOWLEDGE_BASE))
display(EVAL.head(3))

## 4. Базовая линия и система
Обе функции принимают вопрос и возвращают словарь с ответом и любыми полезными для отладки полями.
Именно их вы и замените.

In [ ]:
from rank_bm25 import BM25Okapi
from nltk.stem import SnowballStemmer
_stem = SnowballStemmer("russian")
def _tokens(text):
    return [_stem.stem(w) for w in re.findall(r"\w+", text.lower().replace("ё", "е"))]
_bm25 = BM25Okapi([_tokens(d) for d in KNOWLEDGE_BASE])

def baseline_system(question):
    """Базовая линия: модель отвечает без документов."""
    answer = llm.chat([{"role": "system", "content": "Отвечай кратко, одним предложением."},
                       {"role": "user", "content": question}], max_tokens=96, tag="baseline")
    return {"answer": answer}

def my_system(question, k=3):
    """Система команды. Демо: поиск BM25 + ответ по найденным абзацам со ссылками."""
    scores = _bm25.get_scores(_tokens(question))
    top = np.argsort(-scores)[:k]
    sources = "\n\n".join(f"[{n}] {KNOWLEDGE_BASE[i]}" for n, i in enumerate(top, 1))
    answer = llm.chat([{"role": "system", "content": "Отвечай кратко по фрагментам, ставь ссылки [n]. "
                                                     "Если ответа нет, скажи: «В документах нет ответа»."},
                       {"role": "user", "content": f"Фрагменты:\n{sources}\n\nВопрос: {question}"}],
                      max_tokens=128, tag="system")
    return {"answer": answer, "doc_ids": [int(i) for i in top]}

print(my_system(EVAL.iloc[0]["question"]))

## 5. Прогон и оценка
* **contains_reference** — нормализованный эталон содержится в ответе (грубо, но прозрачно; замените
  метрикой своей задачи: точность класса, F1 полей, доля решённых задач агента и т. п.);
* **judge** — попарное сравнение «система против базовой линии» моделью-судьёй, каждая пара дважды
  с перестановкой; несогласованные вердикты — ничья;
* **latency / cost** — из журнала вызовов.

Модель-судья по умолчанию та же, что отвечает. Если есть доступ к более сильной модели, создайте для судьи
отдельный `LLM(...)` — и обязательно проверьте вручную 10–20 вердиктов.

In [ ]:
def normalize(text):
    return " ".join(re.findall(r"\w+", str(text).lower().replace("ё", "е")))

def run(system_fn, name):
    rows = []
    for _, task in EVAL.iterrows():
        t0 = time.perf_counter()
        try:
            out = system_fn(task["question"]); error = None
        except Exception as exc:
            out, error = {"answer": ""}, f"{type(exc).__name__}: {exc}"
        rows.append({"id": task["id"], "system": name, "answer": out.get("answer", ""), "error": error,
                     "latency_s": time.perf_counter() - t0,
                     "contains_reference": bool(task.get("reference")) and normalize(task["reference"]) in normalize(out.get("answer", ""))})
    return pd.DataFrame(rows)

t0 = time.perf_counter()
runs = pd.concat([run(baseline_system, "baseline"), run(my_system, "system")], ignore_index=True)
print(f"Прогон: {time.perf_counter() - t0:.0f} с")
display(runs.groupby("system")[["contains_reference", "latency_s"]].mean().round(3))

In [ ]:
JUDGE_PROMPT = """Сравни два ответа на вопрос. Важнее всего фактическая правильность, затем полнота и краткость.
Вопрос: {q}
{ref}
Ответ A: {a}
Ответ B: {b}
Какой ответ лучше? Ответь одним символом: A, B или =."""
judge = llm   # замените на LLM(...) с более сильной моделью, если она доступна

def judge_once(question, a, b, reference=""):
    ref = f"Эталонный ответ (для справки): {reference}" if reference else ""
    raw = judge.chat([{"role": "user", "content": JUDGE_PROMPT.format(q=question, ref=ref, a=a, b=b)}],
                     max_tokens=5, tag="judge")
    m = re.search(r"[AB=АВ]", raw.upper())
    return {"A": "A", "А": "A", "B": "B", "В": "B"}.get(m.group(0) if m else "=", "=")

def pairwise(system_name="system", baseline_name="baseline"):
    rows = []
    sys_ans = runs[runs["system"].eq(system_name)].set_index("id")["answer"]
    base_ans = runs[runs["system"].eq(baseline_name)].set_index("id")["answer"]
    for _, task in EVAL.iterrows():
        s, b = sys_ans[task["id"]], base_ans[task["id"]]
        v1 = judge_once(task["question"], s, b, task.get("reference", ""))     # система на месте A
        v2 = judge_once(task["question"], b, s, task.get("reference", ""))     # система на месте B
        win1, win2 = {"A": "system", "B": "baseline", "=": "tie"}[v1], {"A": "baseline", "B": "system", "=": "tie"}[v2]
        verdict = win1 if win1 == win2 else (win1 if win2 == "tie" else win2 if win1 == "tie" else "tie")
        rows.append({"id": task["id"], "verdict": verdict, "consistent": not (v1 == v2 != "=")})
    return pd.DataFrame(rows)

pairs = pairwise()
display(pairs["verdict"].value_counts(normalize=True).round(3).to_frame("доля"))
print(f"Согласованность судьи при перестановке: {pairs['consistent'].mean():.0%}")

## 6. Отчёт
Таблица ниже и `report.json` — основа раздела «Результаты» в README решения и слайда на защите.

In [ ]:
calls = pd.read_json(RUN_DIR / "calls.jsonl", lines=True)
per_system = []
for name in ("baseline", "system"):
    c = calls[calls["tag"].eq(name)]
    r = runs[runs["system"].eq(name)]
    per_system.append({
        "система": name,
        "contains_reference": r["contains_reference"].mean(),
        "p50 задержки, с": r["latency_s"].quantile(0.5),
        "p95 задержки, с": r["latency_s"].quantile(0.95),
        "вход. токенов на запрос": c["input_tokens"].mean(),
        "$ на 1000 запросов": 1000 * c["cost_usd"].sum() / max(1, len(r)),
        "ошибок": int(r["error"].notna().sum()),
    })
report_table = pd.DataFrame(per_system)
win_rate = (pairs["verdict"] == "system").mean(); loss_rate = (pairs["verdict"] == "baseline").mean()
display(report_table.round(4))
print(f"Попарно (система против базовой линии): побед {win_rate:.0%}, поражений {loss_rate:.0%}, "
      f"ничьих {1 - win_rate - loss_rate:.0%}; согласованность судьи {pairs['consistent'].mean():.0%}")

report = {"team": TEAM, "backend": BACKEND, "model": llm.model, "eval_size": len(EVAL),
          "eval_set_sha256": hashlib.sha256((RUN_DIR / "eval_set.jsonl").read_bytes()).hexdigest()[:16],
          "systems": report_table.round(4).to_dict(orient="records"),
          "pairwise": {"win": win_rate, "loss": loss_rate, "tie": 1 - win_rate - loss_rate,
                       "judge_consistency": float(pairs["consistent"].mean())},
          "prices_per_million": {"input": PRICE_IN_PER_M, "output": PRICE_OUT_PER_M}}
(RUN_DIR / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))
runs.merge(EVAL[["id", "question", "reference"]], on="id").to_csv(RUN_DIR / "answers.csv", index=False)
print("Сохранено:", sorted(p.name for p in RUN_DIR.iterdir()))

## 7. Ошибки — материал для защиты
Покажите на защите 2–3 типичные ошибки и что вы с ними сделали. Начните с задач, где система проиграла
базовой линии или не содержит эталона.

In [ ]:
view = (runs.pivot(index="id", columns="system", values="answer")
        .join(EVAL.set_index("id")[["question", "reference"]]).join(pairs.set_index("id")["verdict"]))
display(view[view["verdict"].ne("system")].head(8)[["question", "reference", "baseline", "system", "verdict"]])

## Чек-лист перед защитой
- [ ] Набор проверок зафиксирован (хеш в `report.json`) и не использовался для подбора промптов «на глаз».
- [ ] Есть базовая линия и сравнение с ней по основной метрике и попарно.
- [ ] Посчитаны задержка p50/p95 и стоимость на 1000 запросов.
- [ ] Вручную проверены 10–20 вердиктов судьи.
- [ ] README: как запустить, версии библиотек, данные и их лицензии, ограничения решения.
- [ ] Демонстрация работает с нуля на чистом окружении.